# Global Academy of Technology
### Department of Artificial Intelligence and Machine Learning
## Course: Advance Natural Language Processing (ANLP)
### Experiment 4: Latent Dirichlet Allocation (LDA) for Topic Discovery & Interpretation

---

### **Objective:**
To apply **Latent Dirichlet Allocation (LDA)** on a document collection and interpret the discovered topics using probabilistic and sampling-based methods.

### **Expected Outcomes:**
1. **Preprocess text data:** Tokenization, stopword removal, POS tagging, and lemmatization using `spaCy` and `NLTK`.
2. **Build a document-term representation:** Vocabulary mapping and Document-Term Matrix (DTM) / Bag-of-Words (BoW).
3. **Train an LDA model:** Fit generative probabilistic models using Variational Bayes (Gensim / Scikit-learn).
4. **Interpret topic-word distributions:** Extract Dirichlet parameters $\phi_k$ (topic-word) and $\theta_d$ (document-topic).

### **Deliverables:**
- Preprocessed corpus
- LDA model output (Perplexity & Coherence $C_v$)
- Topic-word distribution lists
- Topic interpretation report
- Visualizations: Word clouds, pyLDAvis intertopic distance, and topic-term distributions

## 1. Import Required Libraries

In [ ]:
import re
import string
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import spacy

import gensim
from gensim.corpora import Dictionary
from gensim.models.ldamodel import LdaModel
from gensim.models.coherencemodel import CoherenceModel
from gensim.models.phrases import Phrases, Phraser

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation
from wordcloud import WordCloud

print(f"Gensim Version: {gensim.__version__}")
print(f"NLTK Version: {nltk.__version__}")
print(f"spaCy Version: {spacy.__version__}")

## 2. Document Collection (Corpus Ingestion)
We load a diverse multi-thematic corpus spanning four distinct technological and scientific disciplines:
1. **Artificial Intelligence & Machine Learning**
2. **Space Science & Astrophysics**
3. **Healthcare & Oncology**
4. **Renewable Energy & Climate Solutions**

In [ ]:
with open('data/sample_tech_science_news.json', 'r', encoding='utf-8') as f:
    raw_data = json.load(f)

df_corpus = pd.DataFrame(raw_data)
print(f"Loaded {len(df_corpus)} documents across {df_corpus['category'].nunique()} reference categories.")
df_corpus[['id', 'title', 'category']].head()

## 3. Preprocessing Text Data
The preprocessing pipeline executes:
- Normalization & regex punctuation removal
- Stopword filtering (NLTK stopwords + domain noise words)
- Part-of-Speech (POS) filtering via `spaCy` (retaining Nouns, Adjectives, Verbs, and Adverbs)
- Lemmatization to base canonical forms
- Collocation / Bigram extraction

In [ ]:
from lda_engine.preprocessor import TextPreprocessor

preprocessor = TextPreprocessor(use_spacy=True, min_word_length=3)
prep_results = preprocessor.preprocess_corpus(df_corpus['text'].tolist(), build_bigrams=True)

tokens = prep_results['tokens']
clean_texts = prep_results['clean_texts']
stats = prep_results['stats']

print("Preprocessing Statistics:")
for k, v in stats.items():
    print(f"  - {k}: {v}")

df_corpus['clean_tokens'] = tokens
df_corpus[['title', 'clean_tokens']].head()

## 4. Build Document-Term Representation (Bag-of-Words & DTM)
Create a Gensim `Dictionary` mapping tokens to unique IDs, and convert documents into sparse Bag-of-Words frequency vectors.

In [ ]:
from lda_engine.vectorizer import DocumentTermMatrix

dtm_builder = DocumentTermMatrix(no_below=1, no_above=0.95)
dtm_stats = dtm_builder.fit_transform(tokens, clean_texts)

print(f"Vocabulary Size: {dtm_stats['vocabulary_size']} unique terms")
print(f"Document-Term Matrix Sparsity: {dtm_stats['matrix_sparsity_percent']}%")
print("\nTop 10 Global Terms by Frequency:")
pd.DataFrame(dtm_stats['top_terms']).head(10)

## 5. Finding Optimal Number of Topics ($k$) using Topic Coherence ($C_v$)
We evaluate topic coherence ($C_v$) over $k \in [2, 6]$ to determine the mathematically optimal topic granularity.

In [ ]:
from lda_engine.evaluation import find_optimal_topics

opt_results = find_optimal_topics(tokens, dtm_builder, min_k=2, max_k=6, passes=10)

plt.figure(figsize=(8, 4.5))
plt.plot(opt_results['k_values'], opt_results['coherence_scores'], marker='o', color='#4f46e5', linewidth=2.5)
plt.title("Topic Coherence ($C_v$) vs Number of Topics ($k$)", fontsize=12, fontweight='bold')
plt.xlabel("Number of Topics ($k$)")
plt.ylabel("Coherence Score ($C_v$)")
plt.grid(True, linestyle='--', alpha=0.6)
plt.axvline(x=opt_results['optimal_k'], color='#ef4444', linestyle=':', label=f"Optimal k = {opt_results['optimal_k']}")
plt.legend()
plt.tight_layout()
plt.show()

print(f"Recommended optimal topics: k = {opt_results['optimal_k']} (C_v = {opt_results['best_coherence']})")

## 6. Train the Latent Dirichlet Allocation (LDA) Model
We train the LDA model using the optimal topic count ($k=4$) with symmetric Dirichlet priors $\alpha$ and $\eta$.

In [ ]:
from lda_engine.lda_model import LDATopicModel
from lda_engine.evaluation import evaluate_topic_coherence

lda = LDATopicModel(num_topics=4, engine="gensim", passes=20, iterations=100)
model_summary = lda.fit(dtm_builder)

coherence = evaluate_topic_coherence(lda.gensim_model, tokens, dtm_builder.dictionary)
print(f"Model Training Complete:")
print(f"  - Engine: {model_summary['engine']}")
print(f"  - Perplexity: {model_summary['perplexity']}")
print(f"  - Topic Coherence (C_v): {coherence.get('c_v')}")
print(f"  - Topic Coherence (U_mass): {coherence.get('u_mass')}")

## 7. Topic-Word Distributions ($\phi_k$)
Inspect top terms and their conditional probability weights for each discovered topic.

In [ ]:
for topic in lda.topic_words:
    print(f"\n--- Topic #{topic['topic_id'] + 1} ---")
    for w in topic['top_words'][:8]:
        print(f"    {w['word']:<20}: {w['weight']:.4f}")

## 8. Visualizations: Word Clouds and Document-Topic Heatmap

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
axes = axes.flatten()

for idx, topic in enumerate(lda.topic_words):
    words_dict = {w['word']: w['weight'] for w in topic['top_words']}
    wc = WordCloud(width=400, height=250, background_color='white', colormap='viridis').generate_from_frequencies(words_dict)
    axes[idx].imshow(wc, interpolation='bilinear')
    axes[idx].set_title(f"Topic #{topic['topic_id'] + 1}", fontsize=12, fontweight='bold')
    axes[idx].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# Document-Topic Heatmap
matrix_theta = np.array([dt['distribution'] for dt in lda.doc_topics])
plt.figure(figsize=(10, 6))
sns.heatmap(matrix_theta, annot=True, fmt='.2f', cmap='Blues', 
            xticklabels=[f"Topic #{i+1}" for i in range(4)],
            yticklabels=[f"Doc #{i+1}" for i in range(len(matrix_theta))])
plt.title("Document-Topic Posterior Distribution ($\theta_{d,k}$)", fontsize=12, fontweight='bold')
plt.xlabel("Latent Topics")
plt.ylabel("Documents")
plt.show()

## 9. Topic Interpretation Report
Synthesize discovered semantic clusters into an automated academic report.

In [ ]:
from lda_engine.interpreter import TopicInterpreter
from IPython.display import display, Markdown

report = TopicInterpreter.generate_interpretation_report(
    topic_words=lda.topic_words,
    doc_topics=lda.doc_topics,
    corpus_stats=dtm_stats,
    model_stats=model_summary,
    coherence_stats=coherence,
    sample_documents=df_corpus['text'].tolist()
)

display(Markdown(report['markdown_report']))

## 10. Real-Time Inference on Unseen Text

In [ ]:
unseen_text = "Deep transformer neural network architectures enable autonomous vehicles to perform real-time computer vision classification and object recognition."
clean_infer_tokens = preprocessor.preprocess_document(unseen_text)
prediction = lda.infer_document(clean_infer_tokens)

print(f"Unseen Text: {unseen_text}")
print(f"Extracted Tokens: {clean_infer_tokens}")
print(f"Dominant Topic: {prediction['dominant_label']} (Probability: {prediction['dominant_probability']*100:.2f}%)")

pd.DataFrame(prediction['distribution'])